Download each CSV file for each month

In [33]:
import requests

for year in range(2026, 2027):
    for month in range(1,13):
        if year == 2026 and month > 7:
            break
        print(f"Downloading data for {year}-{month:02d}...")
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        open(f"ontime_{year}_{month}.zip", "wb").write(r.content)

Import some libraries, including snowflake connector

In [ ]:
import io, gzip, zipfile, snowflake.connector

Create the connection to snowflake and put the credetials in a separate file.
Also created a warehouse in snowflake, a database and a schema to store the data.

In [33]:
import credentials
conn = snowflake.connector.connect(
    user=credentials.credentials['user'],
    password=credentials.credentials['password'],
    account=credentials.credentials['account'],
    warehouse=credentials.credentials['warehouse'],
    database=credentials.credentials['database'],
    schema=credentials.credentials['schema']
)

Before the next step I also created a STAGE inside snowflake to store the stage data or raw data.

Unzip the downloaded files locally and push them to the snowflake staging area called @staging_data. 

In [22]:
import os, re, io, gzip, zipfile
for file in os.listdir():
    print(f"Found {file}")
    if file.startswith("ontime_") and file.endswith(".zip"):
        print(f"Processing {file} ...")
        with open(file, "rb") as f:
            zip_data = f.read()
            with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
                for name in z.namelist():
                    if name.endswith(".csv"):
                        # clean output name: no parentheses, ends in .csv 
                        out = re.sub(r"[()]", "", os.path.basename(name)) + ".gz"
                        with z.open(name) as csv_file, gzip.open(out, "wt", encoding="utf-8", newline="") as dst:
                            for line in io.TextIOWrapper(csv_file, encoding="utf-8", errors="replace", newline=""):
                                line = line.rstrip("\r\n")
                                if line.endswith(","):
                                    line = line[:-1]          # drop only the one trailing comma
                                dst.write(line + "\n")
                        print(f"Uploading {out}")
                        conn.cursor().execute(f"PUT 'file://{out}' @staging_data AUTO_COMPRESS=FALSE OVERWRITE=TRUE")

Processing ontime_2025_8.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_8.csv.gz
Processing ontime_2025_9.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.csv.gz
Processing ontime_2025_4.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_4.csv.gz
Processing ontime_2025_5.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_5.csv.gz
Processing ontime_2025_7.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.csv.gz
Processing ontime_2025_6.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_6.csv.gz
Processing ontime_2025_2.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_2.csv.gz
Processing ontime_2025_10.zip ...
Uploading On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_10.csv.gz
Processing ontime_2025_11.zip ...
Uploading On_Time_Reporting_Carrier_

After we successfully moved the files to snowflake, now we need to remove them locally.

In [ ]:
for file in os.listdir():
    if file.endswith("gz") or file.endswith("zip"):
        print(f"Removing {file} ...")
        os.remove(file)

Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_3.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_12.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_2.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_10.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_6.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_9.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_5.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_7.csv.gz ...
Removing On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2026_4.csv.gz ...


Next step is to create the weather tables for precipitation, temperatures, and wind
But first I am going to clone the dbt-snowflake getting started project to this repo and link github with snowflake account.